<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/RAG_Embedding_Vectordb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Zomato customer support, IPL final night, 9:30 pm.**

Chats jump 10×. The help center has the right answers, but customers never use the words the articles use:

```
 Customer types:  "there is a cockroach in my dal"
 Article title:   "Found something in your food"        ← 0 words in common
```

In [ ]:
!gdown 1qNkpDoaq5vIW1f1NmUmuYf7RTtJFOysn

In [ ]:
# !pip install chromadb

In [ ]:
import string, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
articles = pd.read_csv("zomato_help_center.csv")
docs = (articles["title"] + ". " + articles["text"]).tolist()     # the text we search over

FILLER = set("i me my the a an is was to of and in it on for but no not you your do how what can be "
             "with at from this that are will when we or if they there".split())

def words(text):                                   # lowercase words, without punctuation and filler words
    return {w.strip(string.punctuation) for w in text.lower().split()} - FILLER - {""}

def show(results):                                 # Chroma results -> a small table
    return pd.DataFrame({"id": results["ids"][0],
                         "title": [m["title"] for m in results["metadatas"][0]],
                         "similarity": (1 - np.array(results["distances"][0])).round(2)})

In [ ]:
print(articles.shape)
articles.head()


(30, 5)


,id,category,audience,title,text
0,Z01,orders,all,Order marked delivered but not received,If the app shows your order as delivered but n...
1,Z02,orders,all,Cancelling an order,You can cancel an order for free within 60 sec...
2,Z03,orders,all,Changing the delivery address after ordering,The address cannot be changed after the order ...
3,Z04,orders,all,Missing or wrong items in your order,If an item is missing or a different dish was ...
4,Z05,orders,all,Scheduling an order for later,Choose Schedule at checkout to pick a delivery...


In [ ]:
def keyword_search(query):
    shared = [len(words(query) & words(doc)) for doc in docs]   # words in common with each article

    best = int(np.argmax(shared))
    return articles.loc[[best], ["id", "title"]].assign(shared_words=shared[best])


keyword_search("there is a cockroach in my dal")

,id,title,shared_words
0,Z01,Order marked delivered but not received,0


In [ ]:
keyword_search("delivery guy shouted at me")

,id,title,shared_words
0,Z01,Order marked delivered but not received,1


In [ ]:
keyword_search("order not delivered yet")

,id,title,shared_words
0,Z01,Order marked delivered but not received,2


## Embeddings

In [ ]:
biryani        = np.array([9, 2])
pani_puri      = np.array([8, 4])
gulab_jamun    = np.array([1, 9])
double_biryani = np.array([18, 4])

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
print("biryani vs pani puri     :", cosine(biryani, pani_puri).round(2))
print("biryani vs gulab jamun   :", cosine(biryani, gulab_jamun).round(2))

biryani vs pani puri     : 0.97
biryani vs gulab jamun   : 0.32


In [ ]:
print("biryani vs double biryani   :", cosine(biryani, double_biryani).round(2))

biryani vs double biryani   : 1.0


In [ ]:
rasmalai = np.array([2, 8])
print("rasamali vs pani puri     :", cosine(rasmalai, pani_puri).round(2))
print("rasmalai vs gulab jamun   :", cosine(rasmalai, gulab_jamun).round(2))


rasamali vs pani puri     : 0.65
rasmalai vs gulab jamun   : 0.99


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")      # a pretrained embedding model (90 MB)
vector = model.encode("My food never arrived", normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
print(vector.shape)
print(vector[:5].round(3))


(384,)
[ 0.02  -0.003  0.035  0.072 -0.017]


In [ ]:
np.linalg.norm(vector)

np.float32(1.0)

In [ ]:
pairs = [("My food never arrived", "My order was not delivered"),
         ("The coupon is not working", "The promo code was rejected"),
         ("I want to cancel my order", "I do not want to cancel my order"),
         ("Where is my refund?", "The cat sat on the mat")]

for a, b in pairs:
    va, vb = model.encode([a, b], normalize_embeddings=True)
    print(f"{np.dot(va, vb):.2f}   {a}  |  {b}")

0.54   My food never arrived  |  My order was not delivered
0.52   The coupon is not working  |  The promo code was rejected
0.97   I want to cancel my order  |  I do not want to cancel my order
0.03   Where is my refund?  |  The cat sat on the mat


# Semantic Similarity

In [ ]:
article_vectors = model.encode(docs, normalize_embeddings=True)
print(article_vectors.shape)

(30, 384)


In [ ]:
def semantic_search(query, k = 3):
    q = model.encode(query, normalize_embeddings=True)
    scores = np.dot(article_vectors, q)
    best = scores.argsort()[::-1][:k]
    return articles.loc[best, ["id", "title", "text"]].assign(score=scores[best].round(2))

In [ ]:
semantic_search("there is a cockroach in my food.")

,id,title,text,score
16,Z17,Found something in your food,"If you found a hair, insect or any foreign obj...",0.50
15,Z16,Food arrived cold or spilled,"If your food arrived cold, leaking or spilled,...",0.30
17,Z18,Food tasted bad or was undercooked,Rate the dish and tell us what was wrong from ...,0.26


In [ ]:
semantic_search("My bill did not have the discount applied")

,id,title,text,score
13,Z14,Promo code not applying,"Each promo code has a minimum order value, an ...",0.51
11,Z12,Payment deducted but order not placed,If the amount was debited from your bank but n...,0.50
14,Z15,Downloading your GST invoice,Open the order in Order History and tap Downlo...,0.39


In [ ]:
semantic_search("delivery guy shouted at me.")

,id,title,text,score
9,Z10,Reporting a delivery partner's behaviour,If a delivery partner behaved badly or made yo...,0.37
0,Z01,Order marked delivered but not received,If the app shows your order as delivered but n...,0.32
15,Z16,Food arrived cold or spilled,"If your food arrived cold, leaking or spilled,...",0.32


In [ ]:
semantic_search("who will win the IPL tonight?")

,id,title,text,score
2,Z03,Changing the delivery address after ordering,The address cannot be changed after the order ...,0.07
11,Z12,Payment deducted but order not placed,If the amount was debited from your bank but n...,0.07
10,Z11,When will I get my refund,"Refunds to UPI, cards and net banking take 5 t...",0.05


In [ ]:
def help_desk(query, min_score=0.25):
    search = semantic_search(query, k=1)
    best = search.iloc[0]
    if best["score"] < min_score:
        return "Sorry! Connecting you to a human agent."

    return search # f'{best["title"]}  (score {best["score"]:.2f}) \n '

In [ ]:
help_desk("the offer code is not working.")

,id,title,text,score
13,Z14,Promo code not applying,"Each promo code has a minimum order value, an ...",0.62


In [ ]:
help_desk("who will win the IPL tonight?")

'Sorry! Connecting you to a human agent.'

In [ ]:
help_desk("What's the weather in Mumbai?")

'Sorry! Connecting you to a human agent.'

In [ ]:
# a) 3-word question and a
# b) 40-word article
# go through MiniLM. How many numbers ( embd dim ) does each get?

In [ ]:
from openai import OpenAI

# Initializes client; automatically loads OPENAI_API_KEY from environment variables
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key= api_key)

response = client.embeddings.create(
    model="text-embedding-3-large",
    input="I want to cancel my order"
)
va = response.data[0].embedding


response = client.embeddings.create(
    model="text-embedding-3-large",
    input="I want to keep my order"
)
vb = response.data[0].embedding

cosine(va, vb)

np.float64(0.6627030100109363)

# Vectordb

In [ ]:
30*384 * 4

46080

In [ ]:
# 1000bytes = 1KB
# 1000KB = 1MB
# 1000MB = 1GB
# 1000GB = 1TB

In [ ]:
( 1000000 * 384 * 4 ) / 1e6

1536.0

In [ ]:
(1e9 * 384 * 4 ) / 1e9

1536.0

# VectorDb - ChromaDb

In [ ]:
# client = chromadb.Client()
# # Create a collection
# collection = client.create_collection(name="my_collection")

# # Add documents
# collection.add(
#     documents=["This is a document about cats", "This is a document about cars"],
#     metadatas=[{"category": "pets"}, {"category": "vehicles"}],
#     ids=["id1", "id2"],
# )

# # Query the collection
# results = collection.query(
#     query_texts=["Tell me about tesla flying car"],
#     n_results=1,
# )
# print(results)

In [ ]:
client = chromadb.Client()                                          # a database in memory

minilm = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")


help_db = client.get_or_create_collection("zomato_help", embedding_function=minilm,
                                          configuration={"hnsw": {"space": "cosine"}})
help_db.add(ids=articles["id"].tolist(),
            documents=docs,
            metadatas=articles[["category", "audience", "title"]].to_dict("records"))

help_db.count()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

30

In [ ]:
results = help_db.query(query_texts=["the delivery guy shouted at me"], n_results=3)
show(results)

,id,title,similarity
0,Z10,Reporting a delivery partner's behaviour,0.38
1,Z16,Food arrived cold or spilled,0.34
2,Z01,Order marked delivered but not received,0.33


In [ ]:
question = "how much do I pay for delivery"
print("Everyone:")
display(show(help_db.query(query_texts=[question], n_results=3)))

Everyone:


,id,title,similarity
0,Z30,How delivery fees are calculated,0.63
1,Z26,Free delivery for Gold members,0.54
2,Z13,Paying with cash on delivery,0.50


In [ ]:
print("Gold members only:")
show(help_db.query(query_texts=[question], n_results=3, where={"audience": "gold"}))

Gold members only:


,id,title,similarity
0,Z26,Free delivery for Gold members,0.54
1,Z28,Gold dining discounts,0.16
2,Z27,Cancelling your Gold membership,-0.05


In [ ]:
disk = chromadb.PersistentClient(path="zomato_db")                  # saved in a folder
saved_db = disk.get_or_create_collection("zomato_help", embedding_function=minilm,
                                         configuration={"hnsw": {"space": "cosine"}})

saved_db.add(ids=articles["id"].tolist(), documents=docs,
             metadatas=articles[["category", "audience", "title"]].to_dict("records"))


In [ ]:
saved_db.count()

30

In [ ]:
reopened = chromadb.PersistentClient(path="zomato_db")
reopened = reopened.get_collection("zomato_help", embedding_function=minilm)

In [ ]:
print("Reopened from disk:", reopened.count(), "articles")

Reopened from disk: 30 articles


In [ ]:
tests = [("the delivery boy was very rude", "Z10"),
         ("money got cut from my account but no order", "Z12"),
         ("there is a cockroach in my dal", "Z17"),
         ("my discount coupon is not working", "Z14"),
         ("the soup was not hot", "Z16"),
         ("they forgot my raita", "Z04"),
         ("can I pay in cash", "Z13"),
         ("how do I stop my Gold subscription", "Z27"),
         ("close my account forever", "Z22"),
         ("where is my order, it has been an hour", "Z06")]

In [ ]:
rows = []
for query, answer in tests:
    by_words = keyword_search(query)["id"].iloc[0]

    by_meaning = help_db.query(query_texts=[query], n_results=1)["ids"][0][0]

    rows.append({"query": query, "answer": answer,
                 "keyword": by_words == answer, "semantic": by_meaning == answer})
scorecard = pd.DataFrame(rows)
scorecard

,query,answer,keyword,semantic
0,the delivery boy was very rude,Z10,False,True
1,money got cut from my account but no order,Z12,False,True
2,there is a cockroach in my dal,Z17,False,True
3,my discount coupon is not working,Z14,False,True
4,the soup was not hot,Z16,False,True
5,they forgot my raita,Z04,False,True
6,can I pay in cash,Z13,True,True
7,how do I stop my Gold subscription,Z27,False,True
8,close my account forever,Z22,True,False
9,"where is my order, it has been an hour",Z06,False,False


In [ ]:
show(help_db.query(query_texts=["close my account forever"], n_results=3,  where={"audience": "all"}))


,id,title,similarity
0,Z22,Deleting your Zomato account,0.37
1,Z02,Cancelling an order,0.29
2,Z23,Changing your registered phone number,0.20
